# 02 — Train Stage 1


In [3]:
from pathlib import Path
import subprocess
import sys

import torch
from google.colab import drive


drive.mount("/content/drive")

project_dir = Path("/content/drive/MyDrive/Research/PUMA/PUMA Nuclei Pipeline")

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-r", str(project_dir / "requirements.txt")],
    check=True,
)

sys.path.insert(0, str(project_dir))

from common.settings import paths as get_paths, models, plans

paths = get_paths()
assert torch.cuda.is_available(), "Select a Colab GPU runtime."

print("GPU:", torch.cuda.get_device_name(0))
print("Stage 1 cache:", paths["stage1_cache"])
print("Stage 2 cache:", paths["stage2_cache"])

Mounted at /content/drive
GPU: NVIDIA L4
Stage 1 cache: /content/drive/MyDrive/Research/PUMA/PUMA Nuclei Pipeline/preprocess/stage1
Stage 2 cache: /content/drive/MyDrive/Research/PUMA/PUMA Nuclei Pipeline/preprocess/stage2


## Check code and prepare local cache (if disk allows)


In [4]:
from common.local_cache import local_copy_if_space
from common.run_history import SCHEMA
from stage1.dataset import Cache


assert Cache(paths["stage1_cache"]).meta["num_nuclei"] == 97429

training_cache = local_copy_if_space(
    paths["stage1_cache"],
    "/content/puma_stage1",
)

print("Training schema:", SCHEMA)
print("Training cache:", training_cache)

Copied preprocessing cache to Colab: /content/puma_stage1
Training schema: puma-audited-v6-20261028
Training cache: /content/puma_stage1


## Train all 16 Stage 1 folds or skip verified complete folds


In [ ]:
import gc

from stage1.train import fit_one


EPOCHS = 30

SELECT_JOBS = None

jobs = (
    SELECT_JOBS
    if SELECT_JOBS is not None
    else [(plan, arch, fold) for plan in plans for arch in models for fold in range(4)]
)

for plan, architecture, fold in jobs:
    print(f"STAGE 1 | {plan} | {architecture} | fold {fold}", flush=True)

    fit_one(
        training_cache,
        paths["results"] / "stage1",
        arch=architecture,
        plan=plan,
        fold=fold,
        epochs=EPOCHS,
        tile_size=512,
        overlap=64,
        micro_batch=0,
        effective_batch=16,
        workers=32,
        validate_every=1,
        grad_checkpointing=True,
        device_str="cuda",
    )

    gc.collect()
    torch.cuda.empty_cache()

STAGE 1 | A | convnext_tiny | fold 0
Previous output preserved at /content/drive/MyDrive/Research/PUMA/PUMA Nuclei Pipeline/results/stage1/plan_A/convnext_tiny/fold_0.previous.20261028T165126056130Z; restarting from epoch 1
BACKBONE convnext_tiny: pretrained ImageNet weights=True; this is initialization, NOT loading a trained PUMA checkpoint.
AMP overflow at epoch 1, batch 0; skipped update, loss scale reduced to 32768
Stage1 A/convnext_tiny/fold0 epoch 1/40 loss train/val=2.4357/1.8489 updates=38 grad=9.3606 candidates train/val=561322/189389 predictions train/val=372571/126926 train P/R/F1=0.192/0.979/0.321 val P/R/F1=0.186/0.977/0.313 train-selected threshold=0.0003 NMS=8.0 best=0.3132
Stage1 A/convnext_tiny/fold0 epoch 2/40 loss train/val=1.0599/0.9317 updates=77 grad=3.5300 candidates train/val=316138/106536 predictions train/val=140093/49423 train P/R/F1=0.508/0.973/0.668 val P/R/F1=0.476/0.972/0.640 train-selected threshold=0.001 NMS=8.0 best=0.6396
Stage1 A/convnext_tiny/fold0 

In [5]:
import gc

from stage1.train import fit_one


EPOCHS = 30

SELECT_JOBS = [
    ("B", "convnext_base", fold)
    for fold in range(4)
]

jobs = (
    SELECT_JOBS
    if SELECT_JOBS is not None
    else [(plan, arch, fold) for plan in plans for arch in models for fold in range(4)]
)

for plan, architecture, fold in jobs:
    print(f"STAGE 1 | {plan} | {architecture} | fold {fold}", flush=True)

    fit_one(
        training_cache,
        paths["results"] / "stage1",
        arch=architecture,
        plan=plan,
        fold=fold,
        epochs=EPOCHS,
        tile_size=512,
        overlap=64,
        micro_batch=0,
        effective_batch=16,
        workers=32,
        validate_every=1,
        grad_checkpointing=True,
        device_str="cuda",
    )

    gc.collect()
    torch.cuda.empty_cache()

STAGE 1 | B | convnext_base | fold 0
BACKBONE convnext_base: pretrained ImageNet weights=True; this is initialization, NOT loading a trained PUMA checkpoint.
Downloading: "https://dl.fbaipublicfiles.com/convnext/convnext_base_1k_224_ema.pth" to /root/.cache/torch/hub/checkpoints/convnext_base_1k_224_ema.pth


/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 32 worker processes in total. Our suggested max number of worker in current system is 12, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 32 worker processes in total. Our suggested max number of worker in current system is 12, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


AMP overflow at epoch 1, batch 0; skipped update, loss scale reduced to 32768
AMP overflow at epoch 1, batch 36; skipped update, loss scale reduced to 16384
Stage1 B/convnext_base/fold0 epoch 1/30 loss train/val=2.4672/2.0050 updates=37 grad=8.1982 candidates train/val=644592/217689 predictions train/val=498752/166734 train P/R/F1=0.145/0.989/0.252 val P/R/F1=0.145/0.989/0.253 train-selected threshold=0.0003 NMS=8.0 best=0.2531
Stage1 B/convnext_base/fold0 epoch 2/30 loss train/val=1.0943/0.8386 updates=76 grad=4.4695 candidates train/val=299182/100818 predictions train/val=91349/31092 train P/R/F1=0.749/0.938/0.833 val P/R/F1=0.737/0.937/0.825 train-selected threshold=0.005 NMS=8.0 best=0.8252
Stage1 B/convnext_base/fold0 epoch 3/30 loss train/val=0.5406/0.5067 updates=115 grad=2.5730 candidates train/val=301221/102175 predictions train/val=75878/25478 train P/R/F1=0.895/0.931/0.913 val P/R/F1=0.893/0.929/0.910 train-selected threshold=0.03 NMS=8.0 best=0.9105
Stage1 B/convnext_base/f